# Preprocessing of slide-seq kidney data

In [27]:
import scanpy as sc
import squidpy as sq
import pandas as pd
import numpy as np
import anndata as ad
import os
import gc

In [28]:
path = "/nfs/team361/dj17/MintFlow_2025/slideseq_analysis/"
save_dir = "/nfs/team361/ms83/data/kidney/"

In [29]:
adata = sc.read_h5ad(path + 'Puck_200903_01.h5ad')

In [30]:
adata

AnnData object with n_obs × n_vars = 18535 × 13359
    obs: 'cell_id', 'Study_slice_id', 'NEMO_K_SLICE_ID', 'NEMO_K_PATIENT_ID', 'Corpus_ID', 'Study_patient_id', 'Data_source', 'Platform', 'Tissue_subregion', 'Status', 'Diagnosis', 'Type', 'Age', 'Age_grouping', 'Sex', 'eGFR', 'eGFR_group', 'BMI', 'Obesity_status', 'eGFR baseline', 'eGFR change', 'RCC_Treatment', 'batch', 'NEMO_cell_label', 'NEMO_neighborhood_label', 'tech', 'seed_labels', 'scANVI_pred', 'scANVI_confidence'
    uns: 'leiden', 'neighbors', 'umap'
    obsm: 'cell_emb', 'cell_umap', 'neighborhood_emb', 'neighborhood_umap', 'spatial'
    layers: 'counts'
    obsp: 'connectivities', 'distances'

In [31]:
adata.obs

,cell_id,Study_slice_id,NEMO_K_SLICE_ID,NEMO_K_PATIENT_ID,Corpus_ID,Study_patient_id,Data_source,Platform,Tissue_subregion,Status,...,eGFR baseline,eGFR change,RCC_Treatment,batch,NEMO_cell_label,NEMO_neighborhood_label,tech,seed_labels,scANVI_pred,scANVI_confidence
Puck_200903_01_GSM5554448_Puck_200903_01_CGGAAAGGGAGTCC,GSM5554448_Puck_200903_01_CGGAAAGGGAGTCC,Puck_200903_01,K084,P043,Zero-shot,3579,HuBMAP,Slide-seqv2,Cortex,Healthy,...,na,na,na,Puck_200903_01,3,1,Slide-seqv2,Unknown,Intercalated cell type A,0.974857
Puck_200903_01_GSM5554448_Puck_200903_01_TTCACGGGTAGTGT,GSM5554448_Puck_200903_01_TTCACGGGTAGTGT,Puck_200903_01,K084,P043,Zero-shot,3579,HuBMAP,Slide-seqv2,Cortex,Healthy,...,na,na,na,Puck_200903_01,3,1,Slide-seqv2,Unknown,Distal convoluted tubule 2,0.546630
Puck_200903_01_GSM5554448_Puck_200903_01_GACGGCTGCCACTG,GSM5554448_Puck_200903_01_GACGGCTGCCACTG,Puck_200903_01,K084,P043,Zero-shot,3579,HuBMAP,Slide-seqv2,Cortex,Healthy,...,na,na,na,Puck_200903_01,27,12,Slide-seqv2,Unknown,Proximal tubule segment 3,0.611089
Puck_200903_01_GSM5554448_Puck_200903_01_TGGTGTAGAGAGTC,GSM5554448_Puck_200903_01_TGGTGTAGAGAGTC,Puck_200903_01,K084,P043,Zero-shot,3579,HuBMAP,Slide-seqv2,Cortex,Healthy,...,na,na,na,Puck_200903_01,65,12,Slide-seqv2,Unknown,Medullary thick ascending limb,0.560503
Puck_200903_01_GSM5554448_Puck_200903_01_AACGCCTAACTGTC,GSM5554448_Puck_200903_01_AACGCCTAACTGTC,Puck_200903_01,K084,P043,Zero-shot,3579,HuBMAP,Slide-seqv2,Cortex,Healthy,...,na,na,na,Puck_200903_01,27,4,Slide-seqv2,Unknown,Principal cell,0.374182
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Puck_200903_01_GSM5554448_Puck_200903_01_TATGCCAACTATAA,GSM5554448_Puck_200903_01_TATGCCAACTATAA,Puck_200903_01,K084,P043,Zero-shot,3579,HuBMAP,Slide-seqv2,Cortex,Healthy,...,na,na,na,Puck_200903_01,13,0,Slide-seqv2,Unknown,Neuron,0.270499
Puck_200903_01_GSM5554448_Puck_200903_01_TTGATAAATTTTTT,GSM5554448_Puck_200903_01_TTGATAAATTTTTT,Puck_200903_01,K084,P043,Zero-shot,3579,HuBMAP,Slide-seqv2,Cortex,Healthy,...,na,na,na,Puck_200903_01,4,0,Slide-seqv2,Unknown,Vascular mural cell,0.319980
Puck_200903_01_GSM5554448_Puck_200903_01_CCTTCAGATTTTTT,GSM5554448_Puck_200903_01_CCTTCAGATTTTTT,Puck_200903_01,K084,P043,Zero-shot,3579,HuBMAP,Slide-seqv2,Cortex,Healthy,...,na,na,na,Puck_200903_01,35,3,Slide-seqv2,Unknown,CDH13+ stromal cell,0.310449
Puck_200903_01_GSM5554448_Puck_200903_01_TCGAAAGTAAGACA,GSM5554448_Puck_200903_01_TCGAAAGTAAGACA,Puck_200903_01,K084,P043,Zero-shot,3579,HuBMAP,Slide-seqv2,Cortex,Healthy,...,na,na,na,Puck_200903_01,14,0,Slide-seqv2,Unknown,Principal cell,0.276560


In [32]:
files = [
    "Puck_200903_01.h5ad",
    "Puck_200903_02.h5ad",
    "Puck_200903_03.h5ad",
    "Puck_200903_05.h5ad",
]

adatas = [sc.read_h5ad(path + f) for f in files]

adata = ad.concat(adatas, index_unique=None)

In [33]:
adata.obs

,cell_id,Study_slice_id,NEMO_K_SLICE_ID,NEMO_K_PATIENT_ID,Corpus_ID,Study_patient_id,Data_source,Platform,Tissue_subregion,Status,...,eGFR baseline,eGFR change,RCC_Treatment,batch,NEMO_cell_label,NEMO_neighborhood_label,tech,seed_labels,scANVI_pred,scANVI_confidence
Puck_200903_01_GSM5554448_Puck_200903_01_CGGAAAGGGAGTCC,GSM5554448_Puck_200903_01_CGGAAAGGGAGTCC,Puck_200903_01,K084,P043,Zero-shot,3579,HuBMAP,Slide-seqv2,Cortex,Healthy,...,na,na,na,Puck_200903_01,3,1,Slide-seqv2,Unknown,Intercalated cell type A,0.974857
Puck_200903_01_GSM5554448_Puck_200903_01_TTCACGGGTAGTGT,GSM5554448_Puck_200903_01_TTCACGGGTAGTGT,Puck_200903_01,K084,P043,Zero-shot,3579,HuBMAP,Slide-seqv2,Cortex,Healthy,...,na,na,na,Puck_200903_01,3,1,Slide-seqv2,Unknown,Distal convoluted tubule 2,0.546630
Puck_200903_01_GSM5554448_Puck_200903_01_GACGGCTGCCACTG,GSM5554448_Puck_200903_01_GACGGCTGCCACTG,Puck_200903_01,K084,P043,Zero-shot,3579,HuBMAP,Slide-seqv2,Cortex,Healthy,...,na,na,na,Puck_200903_01,27,12,Slide-seqv2,Unknown,Proximal tubule segment 3,0.611089
Puck_200903_01_GSM5554448_Puck_200903_01_TGGTGTAGAGAGTC,GSM5554448_Puck_200903_01_TGGTGTAGAGAGTC,Puck_200903_01,K084,P043,Zero-shot,3579,HuBMAP,Slide-seqv2,Cortex,Healthy,...,na,na,na,Puck_200903_01,65,12,Slide-seqv2,Unknown,Medullary thick ascending limb,0.560503
Puck_200903_01_GSM5554448_Puck_200903_01_AACGCCTAACTGTC,GSM5554448_Puck_200903_01_AACGCCTAACTGTC,Puck_200903_01,K084,P043,Zero-shot,3579,HuBMAP,Slide-seqv2,Cortex,Healthy,...,na,na,na,Puck_200903_01,27,4,Slide-seqv2,Unknown,Principal cell,0.374182
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Puck_200903_05_GSM5554451_Puck_200903_05_CCGAATAAATATGT,GSM5554451_Puck_200903_05_CCGAATAAATATGT,Puck_200903_05,K087,P043,Zero-shot,3579,HuBMAP,Slide-seqv2,Cortex,Healthy,...,na,na,na,Puck_200903_05,16,1,Slide-seqv2,Unknown,Distal convoluted tubule 2,0.404348
Puck_200903_05_GSM5554451_Puck_200903_05_CTTGACCATTTTTT,GSM5554451_Puck_200903_05_CTTGACCATTTTTT,Puck_200903_05,K087,P043,Zero-shot,3579,HuBMAP,Slide-seqv2,Cortex,Healthy,...,na,na,na,Puck_200903_05,35,0,Slide-seqv2,Unknown,Peritubular capillary,0.424902
Puck_200903_05_GSM5554451_Puck_200903_05_GAGTGGTTTTTTTT,GSM5554451_Puck_200903_05_GAGTGGTTTTTTTT,Puck_200903_05,K087,P043,Zero-shot,3579,HuBMAP,Slide-seqv2,Cortex,Healthy,...,na,na,na,Puck_200903_05,25,12,Slide-seqv2,Unknown,Distal convoluted tubule 2,0.424185
Puck_200903_05_GSM5554451_Puck_200903_05_TCTCGGTACTGCTT,GSM5554451_Puck_200903_05_TCTCGGTACTGCTT,Puck_200903_05,K087,P043,Zero-shot,3579,HuBMAP,Slide-seqv2,Cortex,Healthy,...,na,na,na,Puck_200903_05,16,1,Slide-seqv2,Unknown,Distal convoluted tubule 2,0.626189


In [34]:
adata

AnnData object with n_obs × n_vars = 80499 × 13359
    obs: 'cell_id', 'Study_slice_id', 'NEMO_K_SLICE_ID', 'NEMO_K_PATIENT_ID', 'Corpus_ID', 'Study_patient_id', 'Data_source', 'Platform', 'Tissue_subregion', 'Status', 'Diagnosis', 'Type', 'Age', 'Age_grouping', 'Sex', 'eGFR', 'eGFR_group', 'BMI', 'Obesity_status', 'eGFR baseline', 'eGFR change', 'RCC_Treatment', 'batch', 'NEMO_cell_label', 'NEMO_neighborhood_label', 'tech', 'seed_labels', 'scANVI_pred', 'scANVI_confidence'
    obsm: 'cell_emb', 'cell_umap', 'neighborhood_emb', 'neighborhood_umap', 'spatial'
    layers: 'counts'

In [35]:
if "counts" in adata.layers:
    print("Raw counts are stored in adata.layers['counts']")

Raw counts are stored in adata.layers['counts']


In [36]:
adata.obs["batch"].value_counts()

batch
Puck_200903_05    23137
Puck_200903_03    21357
Puck_200903_01    18535
Puck_200903_02    17470
Name: count, dtype: int64

In [37]:
adata.obs["scANVI_pred"].value_counts()

scANVI_pred
CDH13+ stromal cell               22490
Proximal tubule segment 3         10556
Medullary thick ascending limb     7321
Distal convoluted tubule 2         5768
Connecting tubule                  4768
Podocyte                           4000
Injured proximal tubule            3932
Peritubular capillary              3190
Cortical thick ascending limb      2513
Myofibroblast                      2471
Vascular mural cell                1966
Distal convoluted tubule 1         1960
Proximal tubule segment 2          1813
Principal cell                     1748
Proximal tubule segment 1          1226
Macrophage                         1049
Intercalated cell type A            945
Glomerular endothelial cell         852
Macula densa                        510
Intercalated cell type B            359
Parietal epithelial cell            322
Neuron                              238
Mesangial cell                      151
Descending thin limb                112
CD8+ T cell                 

In [38]:
print(adata.n_obs, adata.n_vars)
print(adata.obs["batch"].value_counts())
print(adata.obsm["spatial"].shape)

80499 13359
batch
Puck_200903_05    23137
Puck_200903_03    21357
Puck_200903_01    18535
Puck_200903_02    17470
Name: count, dtype: int64
(80499, 2)


In [39]:
adata.X = adata.layers["counts"].copy()

In [40]:
sc.pp.filter_cells(adata, min_counts=10)
sc.pp.filter_genes(adata, min_cells=5)

In [41]:
adata

AnnData object with n_obs × n_vars = 80499 × 13359
    obs: 'cell_id', 'Study_slice_id', 'NEMO_K_SLICE_ID', 'NEMO_K_PATIENT_ID', 'Corpus_ID', 'Study_patient_id', 'Data_source', 'Platform', 'Tissue_subregion', 'Status', 'Diagnosis', 'Type', 'Age', 'Age_grouping', 'Sex', 'eGFR', 'eGFR_group', 'BMI', 'Obesity_status', 'eGFR baseline', 'eGFR change', 'RCC_Treatment', 'batch', 'NEMO_cell_label', 'NEMO_neighborhood_label', 'tech', 'seed_labels', 'scANVI_pred', 'scANVI_confidence', 'n_counts'
    var: 'n_cells'
    obsm: 'cell_emb', 'cell_umap', 'neighborhood_emb', 'neighborhood_umap', 'spatial'
    layers: 'counts'

In [42]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [43]:
adata.write_h5ad(save_dir + "adata_kidney_slideseq_all_preprocessed.h5ad")